# Genres of Ugaritic texts

*Hour 2 · ~10 min reading + notebooks `2a` / `2b`*

> **Status:** outline stub.

> **New term?** *TF-IDF, keyword, vector, clustering* and other computational
> terms are unpacked in plain language in [glossary.md](glossary.md).

## The KTU numbering convention
In CAT/KTU, each text has a unique number consisting of a single digit genre code,
followed by a period, followed by one to three more digits. For example:
`KTU 1.14`, `KTU 3.2`, `KTU 4.143`.

The first digit indicates the genre of the text:
- **1.** literary and religious texts — e.g. mythological (the Baal Cycle, Kirta, Aqhat, etc.), sacrifices, calendars, liturgies, omens.
- **2.** letters — diplomatic, administrative, personal
- **3.** legal texts
- **4.** economic or administrative texts — lists, accounts, rations
- **5.** scribal exercises
- **6.** inscriptions on seals, labels, ivories, etc.
- **7.** unclassified texts
- **8.** illegible tablets and uninscribed fragments
- **9.** unpublished texts

## UDB genres

> **Central question for Hour 2:** *How much of the genre is visible from the
> vocabulary alone?* We test this with TF-IDF keywords (`2a`) and with similarity
> + clustering (`2b`), then compare the machine's grouping to the philologists'.

## Caveats to flag before the exercises
- Word **forms vs lemmas** (no morphological normalization in the sample data).
- **Short** and **damaged** texts distort statistics.
- **Genre formulas** can dominate; watch for **false thematic signals**.

# Hour 2 · Keywords and TF-IDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/2a_tfidf_keywords.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F2a_tfidf_keywords.ipynb)


**Goal:** automatically surface the vocabulary *characteristic* of each tablet and genre.

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [1]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. Pick a genre-diverse working set

In [2]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 25]
from collections import Counter
print(len(sample), "tablets:", Counter(t["genre"] for t in sample))

95 tablets: Counter({'letter': 63, 'ritual': 15, 'myth': 12, 'divination': 5})


## 2. Plain word frequencies first

In [3]:
from data.loader import token_counts
token_counts(sample).most_common(15)

[('w', 951),
 ('l', 671),
 ('b', 387),
 ('bʿl', 231),
 ('il', 165),
 ('mlk', 160),
 ('ilm', 153),
 ('k', 139),
 ('š', 127),
 ('rgm', 114),
 ('ym', 108),
 ('bn', 104),
 ('bt', 95),
 ('d', 89),
 ('ʿm', 89)]

## 3. TF-IDF — distinctive words per tablet

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
vec = TfidfVectorizer(token_pattern=r"[^\s]+")
X = vec.fit_transform(docs)
print("matrix (tablets x vocab):", X.shape)

matrix (tablets x vocab): (95, 3256)


## 4. Top keywords per tablet — can you guess the genre?

In [5]:
import numpy as np
feat = np.array(vec.get_feature_names_out())
def top_kw(i, n=6):
    row = X[i].toarray().ravel()
    idx = row.argsort()[::-1][:n]
    return [feat[j] for j in idx if row[j] > 0]
for i, t in enumerate(sample[:15]):
    print(f'{t["ktu"]:7s} [{t["genre"]:11s}] {top_kw(i)}')

1.1     [myth       ] ['il', 'arṣ', 'l', 'w', 'b', 'ṯr']
1.100   [myth       ] ['nḥš', 'lnh', 'ʿqšr', 'nṯk', 'ḥmt', 'mnt']
1.103   [divination ] ['in', 'ḥwt', 'bh', 'w', 'mlkn', 'hyt']
1.105   [ritual     ] ['š', 'ʿlm', 'alp', 'ršp', 'šm', 'ġb']
1.106   [ritual     ] ['gdlt', 'mlk', 'w', 'š', 'šlḥmt', 'l']
1.109   [ritual     ] ['š', 'bʿl', 'ṣpn', 'šlmm', 'w', 'ilib']
1.112   [ritual     ] ['b', 'w', 'ʿšrt', 'btbt', 'šm', 'gṯrm']
1.114   [myth       ] ['il', 'w', 'ʿṯtrt', 'yšt', 'ydʿnn', 'škr']
1.119   [ritual     ] ['bʿl', 'l', 'bt', 'gdlt', 'b', 'ʿz']
1.124   [divination ] ['dtn', 'qḥ', 'mr', 'w', 'mṯpẓ', 'št']
1.130   [ritual     ] ['l', 'š', 'bʿl', 'wš', 'alp', 'kbdm']
1.132   [ritual     ] ['gdlt', 'ḫbtd', 'ṯlnd', 'dqt', 'kmm', 'in']
1.140   [divination ] ['aṯt', 'tld', 'k', 'ḥwt', 'mrḥymlkmlkn', 'ʿḏrt']
1.161   [ritual     ] ['ṯʿy', 'qra', 'w', 'ʿdmt', 'tḥt', 'šlm']
1.163   [divination ] ['hm', 'yrḫ', 'ʿlyh', 'b', 'ṯlṯ', 'pḥm']


## 5. Discussion
- How much of the genre is visible from keywords alone?
- **Caveats:** word *forms* vs lemmas (CUC has no lemmatisation — homographs blur the signal); short/damaged tablets; genre formulas dominating.

> **Production version:** UgaritLab (`generate_stats.py`) computes TF-IDF over the whole corpus and projects it for browsing.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [6]:
# Show more or fewer keywords per tablet, and look at a different slice.
N = 10                       # ← number of keywords
START = 15                   # ← which tablets to inspect (try 0, 30, 45)
for i in range(START, min(START+12, len(sample))):
    print(f'{sample[i]["ktu"]:7s} [{sample[i]["genre"]:11s}] {top_kw(i, N)}')
# Hint: longer keyword lists start to include shared particles — where does the signal fade?

1.164   [divination ] ['w', 'kmm', 'ydbḥ', 'ilib', 'ʿšrh', 'id', 'šlmm', 'šrp', 'ḫmš', 'ḫrṣ']
1.168   [ritual     ] ['kmm', 'yph', 'š', 'w', 'id', 'ḫrṣ', 'npš', 'ʿnt', 'ksp', 'xixx']
1.2     [myth       ] ['nhr', 'ṯpṭ', 'ym', 'zbl', 'l', 'bʿl', 'w', 'mlak', 'bn', 'il']
1.23    [myth       ] ['w', 'il', 'aṯtm', 'b', 'l', 'hm', 'šd', 'mtqtm', 'mštʿltm', 'ylt']
1.24    [myth       ] ['nkl', 'ḫrḫb', 'ašr', 'mznm', 'qẓ', 'kṯrt', 'rḫ', 'yrḫ', 'snnt', 'tzdn']
1.3     [myth       ] ['b', 'l', 'arṣ', 'w', 'bʿl', 'mṯb', 'bt', 'il', 'ʿnt', 'šmm']
1.39    [ritual     ] ['gdlt', 'š', 'dqt', 'ilhm', 'pgr', 'w', 'ṯʿm', 'bʿl', 'il', 'ilt']
1.4     [myth       ] ['b', 'bʿl', 'w', 'aṯrt', 'il', 'aliyn', 'ym', 'rbt', 'l', 'ilm']
1.40    [ritual     ] ['u', 'lp', 'npy', 'nkt', 'ytši', 'bn', 'il', 'ṯʿ', 'hw', 'l']
1.41    [ritual     ] ['gdlt', 'w', 'š', 'ilhm', 'dqt', 'l', 'šlmm', 'b', 'brr', 'ʿṣrm']
1.43    [ritual     ] ['ṯql', 'l', 'ylk', 'gṯrm', 'npš', 'ʿnth', 'ʿntm', 'ap', 'w', 'ḫrṣ']
1.46    [ritual